## core

In [11]:
%pip install pysbd --quiet ##tentei usar esse, mas n suporta portugues bundão - https://github.com/nipunsadvilkar/pySBD/blob/master/pysbd/languages.py

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install nltk --quiet #

Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from difflib import SequenceMatcher

In [2]:
df = pd.read_csv("../datasets/base_df.csv")
df.head()

,texto,rotulo,argumento
0,carnaval em olinda. arrastão monstro. fazuele ...,"Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...
1,carro alegórico da escola de samba grande rio...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o medo ao mencionar que há 'ca...
2,cantor léo apoiando atos antidemocráticos. alg...,"Xingamento-Rotulação,Dúvida,Apelo_ao_Medo-Prec...",O texto menciona 'cantor léo apoiando atos ant...
3,versão 1 ore muito pela cantora anitta! ela es...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza apelo ao medo ao descrever a g...
4,globo já gravou o vídeo de fim de ano com lula...,"Linguagem_Carregada,Apelo_ao_Medo-Preconceito,...","O texto utiliza linguagens carregadas, como ""v..."


## divisão em sentenças

https://www.nltk.org/

https://www.nltk.org/howto/portuguese_en.html

In [3]:
texto0 = str(df.iloc[0]["texto"])
trecho_ate_primeiro_ponto = []
frases = texto0.split(".")
trechos = [f.strip() for f in frases[:10] if f.strip()]

qtd_palavras = [len(f.split()) for f in trechos]

print(qtd_palavras)

[3, 2, 19, 59]


In [4]:
print(trechos[0])
print(trechos[1])
print(trechos[2])
print(trechos[3])

carnaval em olinda
arrastão monstro
fazuele isso foi só um aperitivo do que tá por vir no carnaval país comandado por bandidos, dá nisso
, 172, sexta feira de carnaval, em olinda os pobres meninos do luladrão promovendo um arrastão, nas ruas de olinda, atacando turistas para roubar um ou outro celular e ter um dinheirinho prá tomar uma cervejinha parecem mais uma praga de gafanhotos atacando uma plantação! o brasil está entregue à criminalidade com a proteção do presidente!  o amor venceu


In [5]:
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

MAX_SENTENCAS = 10
MIN_PALAVRAS = 5

In [6]:
df_segmentar = df.copy()

In [7]:
def segmentar_texto(texto: str, min_palavras: int = MIN_PALAVRAS) -> list[str]:
    """
    Descarta frases com menos de MIN_PALAVRAS palavras.
    """
    sentencas = sent_tokenize(str(texto), language="portuguese")
    return [s.strip() for s in sentencas if len(s.split()) >= min_palavras]

In [8]:
df_segmentar["sentencas"] = df_segmentar["texto"].apply(segmentar_texto)

n = df_segmentar["sentencas"].apply(len)
print(f"Sentenças por notícia → média: {n.mean():.1f} | mediana: {n.median():.0f} | máx: {n.max()} | mín: {n.min()}")

Sentenças por notícia → média: 6.6 | mediana: 5 | máx: 52 | mín: 1


In [9]:
df_segmentar.head()

,texto,rotulo,argumento,sentencas
0,carnaval em olinda. arrastão monstro. fazuele ...,"Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...,[fazuele isso foi só um aperitivo do que tá po...
1,carro alegórico da escola de samba grande rio...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o medo ao mencionar que há 'ca...,[carro alegórico da escola de samba grande rio...
2,cantor léo apoiando atos antidemocráticos. alg...,"Xingamento-Rotulação,Dúvida,Apelo_ao_Medo-Prec...",O texto menciona 'cantor léo apoiando atos ant...,"[cantor léo apoiando atos antidemocráticos., a..."
3,versão 1 ore muito pela cantora anitta! ela es...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza apelo ao medo ao descrever a g...,"[versão 1 ore muito pela cantora anitta!, some..."
4,globo já gravou o vídeo de fim de ano com lula...,"Linguagem_Carregada,Apelo_ao_Medo-Preconceito,...","O texto utiliza linguagens carregadas, como ""v...",[globo já gravou o vídeo de fim de ano com lul...


In [10]:
df_segmentar['noticia_id'] = range(len(df_segmentar))

In [11]:
df_segmentar['sentencas'][0]

['fazuele isso foi só um aperitivo do que tá por vir no carnaval país comandado por bandidos, dá nisso.',
 ', 172, sexta feira de carnaval, em olinda os pobres meninos do luladrão promovendo um arrastão, nas ruas de olinda, atacando turistas para roubar um ou outro celular e ter um dinheirinho prá tomar uma cervejinha parecem mais uma praga de gafanhotos atacando uma plantação!',
 'o brasil está entregue à criminalidade com a proteção do presidente!']

In [12]:
df_segmentar["sentencas_A"] = df_segmentar["sentencas"].apply(lambda s: s[:MAX_SENTENCAS])

In [13]:
df_segmentar

,texto,rotulo,argumento,sentencas,noticia_id,sentencas_A
0,carnaval em olinda. arrastão monstro. fazuele ...,"Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...,[fazuele isso foi só um aperitivo do que tá po...,0,[fazuele isso foi só um aperitivo do que tá po...
1,carro alegórico da escola de samba grande rio...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o medo ao mencionar que há 'ca...,[carro alegórico da escola de samba grande rio...,1,[carro alegórico da escola de samba grande rio...
2,cantor léo apoiando atos antidemocráticos. alg...,"Xingamento-Rotulação,Dúvida,Apelo_ao_Medo-Prec...",O texto menciona 'cantor léo apoiando atos ant...,"[cantor léo apoiando atos antidemocráticos., a...",2,"[cantor léo apoiando atos antidemocráticos., a..."
3,versão 1 ore muito pela cantora anitta! ela es...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza apelo ao medo ao descrever a g...,"[versão 1 ore muito pela cantora anitta!, some...",3,"[versão 1 ore muito pela cantora anitta!, some..."
4,globo já gravou o vídeo de fim de ano com lula...,"Linguagem_Carregada,Apelo_ao_Medo-Preconceito,...","O texto utiliza linguagens carregadas, como ""v...",[globo já gravou o vídeo de fim de ano com lul...,4,[globo já gravou o vídeo de fim de ano com lul...
...,...,...,...,...,...,...
1781,"- bom dia, eu to aqui no paraguai. estamos aq...","Apelo_ao_Medo-Preconceito,Repetição,Xingamento...",O texto contém várias técnicas de propaganda. ...,"[- bom dia, eu to aqui no paraguai., estamos a...",1781,"[- bom dia, eu to aqui no paraguai., estamos a..."
1782,pesquisem quem e michael yeadon. e pesquisam o...,"Apelo_ao_Medo-Preconceito,Xingamento-Rotulação...",O texto utiliza o Apelo ao Medo ao sugerir que...,"[pesquisem quem e michael yeadon., e pesquisam...",1782,"[pesquisem quem e michael yeadon., e pesquisam..."
1783,na vacinacao do covid e super importante olhar...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o Apelo ao Medo ao instigar an...,[na vacinacao do covid e super importante olha...,1783,[na vacinacao do covid e super importante olha...
1784,lider indigina morre apos tomar vachina! a mor...,"Apelo_ao_Medo-Preconceito,Culpa_por_Associação...",O texto utiliza apelo ao medo ao sugerir que a...,"[lider indigina morre apos tomar vachina!, a m...",1784,"[lider indigina morre apos tomar vachina!, a m..."


### 10 primeiras

In [14]:
df_A = (
    df_segmentar[["noticia_id", "sentencas_A", "rotulo", "argumento"]]
    .explode("sentencas_A")
    .rename(columns={"sentencas_A": "sentenca"})
    .dropna(subset=["sentenca"])
    .reset_index(drop=True)
)

df_A["pos_sentenca"] = df_A.groupby("noticia_id").cumcount()

In [15]:
print(f"[A] Total de sentenças : {len(df_A)}")
print(f"[A] Média por notícia  : {df_A.groupby('noticia_id').size().mean():.1f}")
print(f"[A] Mínimo por notícia : {df_A.groupby('noticia_id').size().min()}")
print(f"[A] Máximo por notícia : {df_A.groupby('noticia_id').size().max()}")
df_A.head()

[A] Total de sentenças : 9916
[A] Média por notícia  : 5.6
[A] Mínimo por notícia : 1
[A] Máximo por notícia : 10


,noticia_id,sentenca,rotulo,argumento,pos_sentenca
0,0,fazuele isso foi só um aperitivo do que tá por...,"Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...,0
1,0,", 172, sexta feira de carnaval, em olinda os p...","Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...,1
2,0,o brasil está entregue à criminalidade com a p...,"Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...,2
3,1,"carro alegórico da escola de samba grande rio,...","Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o medo ao mencionar que há 'ca...,0
4,1,veja o vídeo é cada abominação nesse carnaval ...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o medo ao mencionar que há 'ca...,1


In [16]:
df_segmentar.drop(columns=["sentencas_A"], inplace=True)

## rotulação

In [17]:
df_segmentar["rotulo"] = df_segmentar["rotulo"].str.split(",")

In [18]:
df_segmentar['rotulo'][0]

['Xingamento-Rotulação',
 'Apelo_ao_Medo-Preconceito',
 'Exagero-Minimização',
 'Linguagem_Carregada',
 'Culpa_por_Associação']

In [19]:
df_rotulos = df_segmentar.explode("rotulo").reset_index(drop=True)
df_rotulos.head()

,texto,rotulo,argumento,sentencas,noticia_id
0,carnaval em olinda. arrastão monstro. fazuele ...,Xingamento-Rotulação,O texto utiliza a rotulação ao chamar Lula de ...,[fazuele isso foi só um aperitivo do que tá po...,0
1,carnaval em olinda. arrastão monstro. fazuele ...,Apelo_ao_Medo-Preconceito,O texto utiliza a rotulação ao chamar Lula de ...,[fazuele isso foi só um aperitivo do que tá po...,0
2,carnaval em olinda. arrastão monstro. fazuele ...,Exagero-Minimização,O texto utiliza a rotulação ao chamar Lula de ...,[fazuele isso foi só um aperitivo do que tá po...,0
3,carnaval em olinda. arrastão monstro. fazuele ...,Linguagem_Carregada,O texto utiliza a rotulação ao chamar Lula de ...,[fazuele isso foi só um aperitivo do que tá po...,0
4,carnaval em olinda. arrastão monstro. fazuele ...,Culpa_por_Associação,O texto utiliza a rotulação ao chamar Lula de ...,[fazuele isso foi só um aperitivo do que tá po...,0


In [22]:
df_rotulos['rotulo'].unique()

array(['Xingamento-Rotulação', 'Apelo_ao_Medo-Preconceito',
       'Exagero-Minimização', 'Linguagem_Carregada',
       'Culpa_por_Associação', 'Dúvida', 'Repetição',
       'Apelo_à_Popularidade', 'Espantalho', 'Apelo_à_Bandeira',
       'Apelo_à_Autoridade', 'Simplificação_Causal',
       'Falso_Dilema-Sem_Escolha', 'Slogans', 'Apelo_à_Hipocrisia',
       'Nenhuma', 'Ofuscação-Vagueza-Confusão', 'Arenque_Vermelho',
       'Encerrador_de_Debate', 'Apelo_à_Authoridade', 'Whataboutismo',
       'Oportunidade de Compartilhamento', 'Urgência',
       'Lingüagem_Carregada', 'Linguaem_Carregada', 'Urgente',
       'O queaboutismo', 'Oportunismo', 'Apelo_À_Popularidade',
       'Lingua_Carregada', 'Apelo_À_Bandeira', 'Luanguage_Carregada',
       'Apelo à Autoridade', 'Orenque_Vermelho'], dtype=object)

In [23]:
rotulo_counts = df_rotulos['rotulo'].value_counts()
print(rotulo_counts)

rotulo
Apelo_ao_Medo-Preconceito           1232
Linguagem_Carregada                 1049
Repetição                            785
Xingamento-Rotulação                 718
Dúvida                               615
Exagero-Minimização                  581
Culpa_por_Associação                 378
Apelo_à_Popularidade                 306
Apelo_à_Autoridade                   199
Apelo_à_Bandeira                     174
Simplificação_Causal                 156
Ofuscação-Vagueza-Confusão           153
Apelo_à_Hipocrisia                   133
Slogans                              111
Encerrador_de_Debate                  87
Arenque_Vermelho                      66
Falso_Dilema-Sem_Escolha              63
Nenhuma                               45
Espantalho                            23
Whataboutismo                         21
Apelo_à_Authoridade                    3
Oportunidade de Compartilhamento       1
Urgência                               1
Lingüagem_Carregada                    1
Linguaem_

refazer o labling com agno com claude